# DDR 2011 — Geração de Arquivo XML Regulatório

Este notebook é a etapa final do pipeline DDR. Lê a planilha já atualizada pelos scripts `cotacao_dolar.py` e `multip_13804.py`, extrai os valores da última data-base válida e gera o XML no formato exigido pelo BACEN.

> **Nota:** Os dados exibidos nas células de output são fictícios, gerados para fins de demonstração.

## 1. Dependências

In [ ]:
# pip install pandas openpyxl lxml

import pandas as pd
from lxml import etree
from pathlib import Path
import zipfile

# Configurações — ajuste conforme o ambiente
CAMINHO_PLANILHA = "caminho/para/DDR.xlsx"
ABA_PLANILHA     = "Arquivos 2026"
OUTPUT_DIR       = "./output"

# Dados do responsável pelo envio
CNPJ_INSTITUICAO  = "00000000"
CODIGO_DOCUMENTO  = "2011"
TIPO_ENVIO        = "I"
NOME_RESPONSAVEL  = "Nome do Responsável"
FONE_RESPONSAVEL  = "1100000000"
EMAIL_RESPONSAVEL = "responsavel@instituicao.com.br"

## 2. Leitura e Tratamento da Planilha

A planilha pode conter múltiplos blocos mensais numa mesma aba. O código detecta cada bloco pelo cabeçalho no formato `MM/YYYY` e seleciona automaticamente o mês mais recente.

In [ ]:
from io import StringIO

# Dados fictícios representando a estrutura esperada da planilha
# Em producao, substitua por: df_raw = pd.read_excel(CAMINHO_PLANILHA, header=None, sheet_name=ABA_PLANILHA)
mock_csv = """
Data-base,Saldo Extrato,Deposito NY,Fator F,Fator Incorp.,Multi. M,CVA
2025-01-02,16766.08,91160.53,22,100,351,1378795.19
2025-01-03,16736.45,90964.28,22,100,348,1378795.19
2025-01-06,15964.60,85881.57,22,100,333,1081788.10
2025-01-07,,0.00,22,100,0,1081788.10
""".strip()

df_mes = pd.read_csv(StringIO(mock_csv))
df_mes["Data-base"] = pd.to_datetime(df_mes["Data-base"]).dt.strftime("%Y-%m-%d")
df_mes["Deposito NY"] = pd.to_numeric(df_mes["Deposito NY"], errors="coerce").round(2)
df_mes["CVA"]         = pd.to_numeric(df_mes["CVA"], errors="coerce").round(2)

print(df_mes)

## 3. Extração da Última Data-base Válida

In [ ]:
df_mes.columns = df_mes.columns.str.strip()
df_mes['Saldo Extrato'] = df_mes['Saldo Extrato'].replace(['', ' ', '-', 'NULL'], pd.NA)

ultima_linha = df_mes[df_mes['Saldo Extrato'].notna()].iloc[-1]
linha_dict   = ultima_linha.to_dict()

print("Valores da ultima data-base valida:")
for k, v in linha_dict.items():
    print(f"  {k}: {v}")

## 4. Geração do XML DDR 2011

Estrutura do XML:
```
documentoDDR
  ├── parametros
  │     ├── parametro (codigo 31 — nome responsável)
  │     ├── parametro (codigo 32 — telefone)
  │     └── parametro (codigo 33 — email)
  └── contas
        ├── conta 111000 (Deposito NY)
        │     └── detalhamentosDDR
        │           └── detalhamentoDDR
        │                 ├── detalhe (elemento 83 — moeda)
        │                 └── detalhe (elemento 84 — tipo)
        ├── conta 310105 (Fator F)
        ├── conta 410100 (Fator Incorp.)
        ├── conta 410101 (Multiplicador M)
        └── conta 411000 (CVA)
```

In [ ]:
documento = etree.Element(
    'documentoDDR',
    cnpj=CNPJ_INSTITUICAO,
    dataBase=str(linha_dict['Data-base']),
    codigoDocumento=CODIGO_DOCUMENTO,
    tipoEnvio=TIPO_ENVIO
)

# Parametros de contato
parametros = etree.SubElement(documento, 'parametros')
for codigo, valor in [
    ("31", NOME_RESPONSAVEL),
    ("32", FONE_RESPONSAVEL),
    ("33", EMAIL_RESPONSAVEL),
]:
    p = etree.SubElement(parametros, 'parametro',
                         codigoParametro=codigo, valorParametro=valor)
    p.text = ''

# Contas
contas = etree.SubElement(documento, 'contas')

# Conta 111000 — Deposito NY (com detalhamento)
conta = etree.SubElement(contas, 'conta',
    codigoConta='111000',
    valorConta=str(linha_dict['Deposito NY'])
)
det_ddrs = etree.SubElement(conta, 'detalhamentosDDR')
det_ddr  = etree.SubElement(det_ddrs, 'detalhamentoDDR',
    valorDetalhe=str(linha_dict['Deposito NY'])
)
etree.SubElement(det_ddr, 'detalhe', codigoElemento='83', valorElemento='USD')
etree.SubElement(det_ddr, 'detalhe', codigoElemento='84', valorElemento='1')

# Demais contas
for codigo, campo in [
    ('310105', 'Fator F'),
    ('410100', 'Fator Incorp.'),
    ('410101', 'Multi. M'),
    ('411000', 'CVA'),
]:
    etree.SubElement(contas, 'conta',
        codigoConta=codigo,
        valorConta=str(linha_dict[campo])
    )

print("XML montado com sucesso.")

## 5. Exportação — XML compactado em ZIP

In [ ]:
import os

data_base     = pd.to_datetime(linha_dict['Data-base'], format='%Y-%m-%d', errors='coerce')
data_arquivo  = linha_dict['Data-base'].replace('-', '')
nome_xml      = f"01{CODIGO_DOCUMENTO}_{data_arquivo}.xml"

# Organiza em subpasta por ano_mes
nome_pasta = data_base.strftime('%Y_%m') if pd.notna(data_base) else "sem_data"
pasta_final = Path(OUTPUT_DIR) / nome_pasta
pasta_final.mkdir(parents=True, exist_ok=True)

caminho_xml = pasta_final / nome_xml

# Grava o XML
with open(caminho_xml, "wb") as f:
    f.write(etree.tostring(
        documento,
        encoding="ISO-8859-1",
        pretty_print=True,
        xml_declaration=True
    ))

# Compacta e remove o XML original
zip_path = pasta_final / nome_xml.replace('.xml', '.zip')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(caminho_xml, arcname=nome_xml)
caminho_xml.unlink()

print(f"Arquivo gerado: {zip_path}")